# Event Extraction Pipeline

This notebook constructs fixed-horizon AAPL event-window inputs for the event-driven Monte Carlo price model.

It can be run in two modes:

1. Default mode: use saved attribution outputs in `data/aapl_h5_sample_outputs.jsonl`
2. Optional API mode: rerun non-earnings event attribution through the OpenAI API

The public repository is intended to run in default mode without submitting new API jobs.

In [1]:
#construct data structure for windows

from dataclasses import dataclass
import pandas as pd
from typing import Optional

@dataclass
class TrajectoryWindow:
    start_idx: int
    end_idx: int
    start_date: pd.Timestamp
    end_date: pd.Timestamp
    horizon: int
    start_price: float
    end_price: float
    simple_return: float
    abs_return: float
    llm_true_agreement: Optional[float] = None
    llm_category: Optional[str] = None
    llm_category_agreement: Optional[float] = None
    llm_note: Optional[str] = None

#import data
import yfinance as yf

ticker = "AAPL"

data = yf.download(ticker, start="2017-01-01", end="2024-01-01", progress=False)

prices = data["Close"][ticker]
price_array = prices.to_numpy()
dates = prices.index

T = len(price_array)

In [2]:
t = yf.Ticker(ticker)

# get a large enough history
earn = t.get_earnings_dates(limit=100)



# restrict to your price sample range
start = pd.Timestamp("2018-01-01")
end = pd.Timestamp("2024-01-01")

earn_dates = pd.to_datetime(earn.index).tz_localize(None).normalize()
earn_dates = earn_dates[(earn_dates >= start) & (earn_dates <= end)]

# remove duplicates and sort

cleaned = []

for d in sorted(earn_dates):
    if not cleaned:
        cleaned.append(d)
    elif (d - cleaned[-1]).days <= 1:
        continue  # ignore later duplicate, keep earliest
    else:
        cleaned.append(d)

earn_dates = pd.DatetimeIndex(cleaned)
earn_dates = pd.DatetimeIndex(sorted(set(earn_dates)))

price_index = prices.index.normalize()


H = 5

earn_windows = []

for d in earn_dates:
    d = pd.Timestamp(d).normalize()

    pos = int(price_index.searchsorted(d, side="left"))
    start_idx = pos - 1

    if start_idx < 0 or start_idx + H >= len(price_index):
        continue

    end_idx = start_idx + H

    earn_windows.append(
        TrajectoryWindow(
            start_idx=start_idx,
            end_idx=end_idx,
            start_date=price_index[start_idx],
            end_date=price_index[end_idx],
            horizon=H,
            start_price=float(price_array[start_idx]),
            end_price=float(price_array[end_idx]),
            simple_return=float(price_array[end_idx] / price_array[start_idx] - 1.0),
            abs_return=float(abs(price_array[end_idx] / price_array[start_idx] - 1.0)),
        )
    )



In [3]:
import numpy as np




windows = []

for start_idx in range(T - H):
    end_idx = start_idx + H

    start_price = price_array[start_idx]
    end_price = price_array[end_idx]

    if np.isnan(start_price) or np.isnan(end_price):
        continue

    simple_return = end_price / start_price - 1.0

    windows.append(
        TrajectoryWindow(
            start_idx=start_idx,
            end_idx=end_idx,
            start_date=dates[start_idx],
            end_date=dates[end_idx],
            horizon=H,
            start_price=float(start_price),
            end_price=float(end_price),
            simple_return=float(simple_return),
            abs_return=float(abs(simple_return)),
        )
    )





In [4]:

def overlaps(w1, w2):
    w1_start = w1.start_idx + 1
    w1_end = w1.end_idx

    w2_start = w2.start_idx + 1
    w2_end = w2.end_idx

    return not (w1_end < w2_start or w2_end < w1_start)

abs_returns = np.array([w.abs_return for w in windows])


lookback = 252
percentile = 80

filtered_windows = []

for i, w in enumerate(windows):
    if i < lookback:
        continue

    local_abs = np.array([x.abs_return for x in windows[i - lookback:i]])
    local_threshold = np.percentile(local_abs, percentile)

    if w.abs_return >= local_threshold:
        filtered_windows.append(w)


non_earn_candidates = []

for w in filtered_windows:
    overlaps_any_earn = False

    for e in earn_windows:
        if overlaps(w, e):
            overlaps_any_earn = True
            break

    if not overlaps_any_earn:
        non_earn_candidates.append(w)

non_earn_candidates = sorted(
    non_earn_candidates,
    key=lambda w: (-w.abs_return, w.start_idx, w.end_idx)
)

selected = []
for w in non_earn_candidates:
    if not any(overlaps(w, s) for s in selected):
        selected.append(w)

selected = sorted(selected, key=lambda w: w.start_idx)


## Optional API Attribution

The following cells show how attribution outputs can be generated through the OpenAI API. They are not required for the repository demo. Saved attribution outputs are included in `data/`, so the downstream filtering step can be run without new API calls.

In [5]:
# Optional API attribution setup

import json
import time
from pathlib import Path


# ----------------------------
# Config
# ----------------------------
RUN_NAME = "aapl_event_classification_batch_v1"
JSONL_PATH = f"{RUN_NAME}.jsonl"
META_PATH = f"{RUN_NAME}_meta.json"
POLL_SECONDS = 30
REPEATS = 5

import os

RUN_API_ATTRIBUTION = False

if RUN_API_ATTRIBUTION:
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    
    
# ----------------------------
# Prompt
# ----------------------------
BASE_PROMPT = """
Classify whether the AAPL price move in the given window is attributable to a single dominant non-earnings event.

Use web search before deciding.
Only consider candidate events that occur:

- within the window, OR
- on a non-trading day immediately preceding the window, OR
- after the market close on the immediately preceding trading day

Treat events announced after the close of the preceding trading day as occurring at the start of the next trading day.

Do NOT consider events that occurred earlier on the immediately preceding trading day (before or during that trading session), as those would already have been reflected in prices prior to the window.

Discard candidates that do not directly affect at least one of:
- product demand
- supply chain / production
- pricing / margins
- regulation
- strategy

An event qualifies only if it introduces new information that changes
the expected medium-term trajectory of the company beyond the window.
Exclude pure volatility, liquidity, or sentiment-driven repricing or continued reaction to previously known information.


Exclude:
- shareholder disclosures
- analyst upgrades/downgrades
- price targets
- index rebalancing
- generic commentary
- sentiment-only events

For each remaining candidate, score:
- temporal alignment
- economic relevance
- magnitude plausibility

score = min(temporal, relevance, magnitude)

Return true only if:
- one dominant candidate exists
- score >= 0.7

Otherwise return false.

If true, assign exactly one category:
- Product / Commercial: direct impact on demand, supply, production, or revenue expectations
- Regulatory / Legal: government policy, tariffs, antitrust, legislation, court rulings
- Strategic / Management: long-term initiatives or business direction without immediate operational impact
- Other Non-Earnings: macro or market-wide forces not specific to Apple

Rules:
- prefer firm-specific over macro
- if both regulatory and macro apply, choose Regulatory / Legal
- if operational impact is immediate, choose Product / Commercial over Strategic / Management
- use Other Non-Earnings only if no firm-specific or regulatory driver qualifies

OUTPUT FORMAT

Return EXACTLY one JSON object with these keys and no others:

{
  "start_date": "<copy input start_date>",
  "end_date": "<copy input end_date>",
  "event_attributable": true or false,
  "category": "<one allowed category or empty string>",
  "notes": "<max 12 words, short label only, no explanation>"
}

Output rules:
- if event_attributable is false, set category="" and notes=""
- notes must be a short label, not an explanation
- do not include dates, scores, source names, percentages, or justification in notes
- use lowercase true/false
- output JSON only
- no extra text
""".strip()

def make_prompt(start_date: str, end_date: str):
    return [
        {
            "role": "system",
            "content": BASE_PROMPT.strip()
        },
        {
            "role": "user",
            "content": f"""WINDOW

start_date: {start_date}
end_date: {end_date}"""
        }
    ]


# ----------------------------
# Helper functions for later reuse in same cell
# ----------------------------
def load_meta(meta_path=META_PATH):
    with open(meta_path, "r", encoding="utf-8") as f:
        return json.load(f)

def retrieve_batch_from_saved_meta(meta_path=META_PATH):
    meta = load_meta(meta_path)
    return client.batches.retrieve(meta["batch_id"])

def wait_for_batch_from_saved_meta(meta_path=META_PATH, poll_seconds=POLL_SECONDS):
    meta = load_meta(meta_path)
    batch_id = meta["batch_id"]

    while True:
        b = client.batches.retrieve(batch_id)
        print("status:", b.status, "| request_counts:", getattr(b, "request_counts", None))

        if b.status in {"completed", "failed", "expired", "cancelled"}:
            return b

        time.sleep(poll_seconds)

def download_text_file(file_id, out_path=None):
    text = client.files.content(file_id).text
    if out_path is not None:
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(text)
    return text

def parse_batch_output(output_text: str):
    parsed = {}
    for line in output_text.splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        cid = obj["custom_id"]

        body = obj.get("response", {}).get("body", {})
        out_text = body.get("output_text")

        if out_text is None:
            parsed[cid] = {"raw": obj}
            continue

        try:
            parsed[cid] = json.loads(out_text)
        except json.JSONDecodeError:
            parsed[cid] = {"raw_output_text": out_text}
    return parsed

def parse_batch_errors(error_text: str):
    parsed = {}
    for line in error_text.splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        parsed[obj["custom_id"]] = obj.get("error")
    return parsed

def fetch_results_from_saved_meta(meta_path=META_PATH):
    meta = load_meta(meta_path)
    b = client.batches.retrieve(meta["batch_id"])

    if b.status != "completed":
        raise RuntimeError(f"Batch not completed. Current status: {b.status}")

    results = {}
    errors = {}

    if b.output_file_id:
        output_text = download_text_file(b.output_file_id, f"{meta['run_name']}_output.jsonl")
        results = parse_batch_output(output_text)

    if b.error_file_id:
        error_text = download_text_file(b.error_file_id, f"{meta['run_name']}_errors.jsonl")
        errors = parse_batch_errors(error_text)

    indexed = []
    req_map = {r["custom_id"]: r for r in meta["requests"]}

    for cid, req in req_map.items():
        indexed.append({
            "custom_id": cid,
            "start_date": req["start_date"],
            "end_date": req["end_date"],
            "result": results.get(cid),
            "error": errors.get(cid)
        })

    return indexed, b

In [6]:
MASTER_META_PATH = META_PATH.replace(".json", "_all_chunks.json")

In [7]:
import time
import json
import pandas as pd
if RUN_API_ATTRIBUTION:



    CHUNK_SIZE = 6
    POLL_INTERVAL = 30  # seconds


    def wait_for_capacity():
        while True:
            batches = client.batches.list()

            active = [
                b for b in batches.data
                if b.status in ("in_progress", "finalizing")
            ]

            if not active:
                return

            print(f"Waiting... {len(active)} active batches")
            time.sleep(POLL_INTERVAL)


    def wait_for_batch(batch_id):
        while True:
            batch = client.batches.retrieve(batch_id)
            print(f"Batch {batch_id} status: {batch.status}")

            if batch.status in ("completed", "failed", "cancelled"):
                return batch

            time.sleep(POLL_INTERVAL)


    all_batches = []
    all_windows = selected

    for chunk_idx in range(0, len(all_windows), CHUNK_SIZE):
        chunk = all_windows[chunk_idx:chunk_idx + CHUNK_SIZE]
        chunk_num = chunk_idx // CHUNK_SIZE

        wait_for_capacity()  # Wait for active batches to finish before submitting the next chunk.

        JSONL_CHUNK = JSONL_PATH.replace(".jsonl", f"_chunk_{chunk_num:02d}.jsonl")
        META_CHUNK = META_PATH.replace(".json", f"_chunk_{chunk_num:02d}.json")

        requests = []

        with open(JSONL_CHUNK, "w", encoding="utf-8") as f:
            for local_i, w in enumerate(chunk):
                global_i = chunk_idx + local_i

                start_date = str(w.start_date + pd.Timedelta(days=1))
                end_date = str(w.end_date)

                for j in range(REPEATS):
                    req = {
                        "custom_id": f"window-{global_i:03d}-run-{j}",
                        "method": "POST",
                        "url": "/v1/responses",
                        "body": {
                            "model": "o3",
                            "tools": [{"type": "web_search"}],
                            "reasoning": {"effort": "medium"},
                            "input": make_prompt(start_date, end_date)
                        }
                    }
                    f.write(json.dumps(req) + "\n")

                    requests.append({
                        "custom_id": req["custom_id"],
                        "window_index": global_i,
                        "run": j,
                        "start_date": start_date,
                        "end_date": end_date
                    })

        print(f"Wrote {JSONL_CHUNK}")

        # upload
        with open(JSONL_CHUNK, "rb") as f:
            input_file = client.files.create(file=f, purpose="batch")

        # create batch
        batch = client.batches.create(
            input_file_id=input_file.id,
            endpoint="/v1/responses",
            completion_window="24h",
            metadata={"run_name": RUN_NAME, "chunk": str(chunk_num)}
        )
        print("Submitted batch:", batch.id)

        # 🔑 wait until this batch finishes before next
        final_batch = wait_for_batch(batch.id)

        # save metadata
        meta = {
            "run_name": RUN_NAME,
            "chunk_num": chunk_num,
            "batch_id": batch.id,
            "status": final_batch.status,
            "input_file_id": input_file.id,
            "requests": requests
        }

        with open(META_CHUNK, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

        all_batches.append(meta)
    MASTER_META_PATH = META_PATH.replace(".json", "_all_chunks.json")
    with open(MASTER_META_PATH, "w", encoding="utf-8") as f:
        json.dump(all_batches, f, indent=2)
    print("All batches complete.")



In [8]:
from pathlib import Path
OUTPUT_PATH = Path("data/aapl_h5_sample_outputs.jsonl")

if RUN_API_ATTRIBUTION:

    

    with open(MASTER_META_PATH, "r", encoding="utf-8") as f:
        all_batches = json.load(f)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as out:
        for meta in all_batches:
            batch_id = meta["batch_id"]
            batch = client.batches.retrieve(batch_id)

            if batch.status != "completed":
                print(f"Skipping batch {batch_id} (status={batch.status})")
                continue

            content = client.files.content(batch.output_file_id).text

            for line in content.splitlines():
                obj = json.loads(line)
                response_output = obj["response"]["body"]["output"]

                for item in response_output:
                    if item["type"] == "message":
                        text = item["content"][0]["text"]
                        out.write(text.strip() + "\n")

    print(f"Saved outputs to {OUTPUT_PATH}")

In [9]:
from collections import defaultdict, Counter



def build_llmselected(selected, json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        text = f.read()

    decoder = json.JSONDecoder()
    i = 0
    grouped = defaultdict(list)

    # parse JSON objects
    while i < len(text):
        while i < len(text) and text[i].isspace():
            i += 1
        if i >= len(text):
            break

        obj, j = decoder.raw_decode(text, i)
        i = j

        key = (obj["start_date"], obj["end_date"])
        grouped[key].append(obj)

    llmselected = []

    for w in selected:
        
        key = (str(w.start_date + pd.Timedelta(days=1)), str(w.end_date))
        rows = grouped.get(key, [])

        if not rows:
            continue

        n = len(rows)
        true_rows = [r for r in rows if r["event_attributable"]]
        true_n = len(true_rows)
        # count categories across ALL queries (only TRUE rows have categories anyway)
        category_counts = Counter(
            r["category"] for r in rows if r["category"] != ""
        )

        if not category_counts:
            continue

        category, count = category_counts.most_common(1)[0]
        

        # majority over all queries
        if count > n / 2:
            w.llm_category = category
            w.llm_category_agreement = count / true_n
            w.llm_true_agreement = true_n / n
            llmselected.append(w)
        candidate_notes = [
            r["notes"]
            for r in true_rows
            if r["category"] == category and r["notes"] != ""
        ]

        w.llm_note = max(candidate_notes, key=len) if candidate_notes else ""
    return llmselected
llmselected=build_llmselected(selected, OUTPUT_PATH)

In [10]:
import json
pathllm="data/aapl_h5_selected_events.json"
pathearning="data/aapl_h5_earnings_events.json"
def save_windows(llmselected, path):
    data = []

    for w in llmselected:
        data.append({
            "start_idx": w.start_idx,
            "end_idx": w.end_idx,
            "start_date": str(w.start_date),
            "end_date": str(w.end_date),
            "horizon": w.horizon,
            "start_price": w.start_price,
            "end_price": w.end_price,
            "simple_return": w.simple_return,
            "abs_return": w.abs_return,
            "llm_true_agreement": getattr(w, "llm_true_agreement", None),
            "llm_category": getattr(w, "llm_category", None),
            "llm_category_agreement": getattr(w, "llm_category_agreement", None),
            "llm_note": getattr(w, "llm_note", None),
        })

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
save_windows(llmselected,pathllm)
save_windows(earn_windows,pathearning)